In [3]:
"""
Model diagnostics and robustness checks, run after 02_model_comparison.py.

Covers:
  1. Hyperparameter tuning (RandomizedSearchCV for RF and XGBoost)
  2. Robustness analysis (repeated stratified CV -> distribution of macro F1)
  3. SHAP dependence plots for top features
  4. Calibration analysis (reliability diagrams, Brier score, per-class)
  5. Feature importance consistency (Gini vs. permutation vs. SHAP)
  6. Error analysis (confusion pairs, subgroup error rates)
  7. Ablation study (feature-group leave-one-out and single-group-only)

Run: python 05_model_diagnostics.py
Requires: pandas, numpy, scikit-learn, xgboost, shap, matplotlib, scipy
Input: findex_pakistan_features_target.csv (output of script 01)
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import (train_test_split, RandomizedSearchCV,
                                      StratifiedKFold, RepeatedStratifiedKFold,
                                      cross_val_score)
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.calibration import calibration_curve
from sklearn.metrics import (f1_score, accuracy_score, brier_score_loss,
                              confusion_matrix)
from xgboost import XGBClassifier
from scipy.stats import spearmanr
import shap

RANDOM_STATE = 42
CSV_PATH = "data/findex_pakistan_features_target.csv"
TIER_NAMES = ["Excluded", "Basic", "Digitally Engaged", "Advanced"]

In [4]:
# ---------------------------------------------------------------------
# 1. Load data, rebuild the same leakage-safe feature set as script 02
# ---------------------------------------------------------------------
data = pd.read_csv(CSV_PATH)
drop_cols = ["adoption_tier", "adoption_tier_label",
             "account", "dig_account", "anydigpayment", "saved", "fin22a",
             "account_fin", "account_mob"]
X = data.drop(columns=drop_cols)
y = data["adoption_tier"]

feature_groups = {
    "demographics": ["female", "age", "educ", "inc_q", "in_workforce", "is_rural"],
    "access": ["con1", "internet_use"],
    "behaviour": ["merchantpay_dig", "fin17b", "fin17c", "borrowed", "fin22a_1"],
    "resilience": ["fin24a", "fin24b"],
}

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
)
print(f"Loaded {data.shape[0]} rows. Train: {len(X_train)}, Test: {len(X_test)}")

Loaded 1000 rows. Train: 750, Test: 250


In [5]:
# =======================================================================
# SECTION 1: HYPERPARAMETER TUNING
# =======================================================================
print("\n" + "=" * 70)
print("1. HYPERPARAMETER TUNING (RandomizedSearchCV, 5-fold stratified CV)")
print("=" * 70)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

rf_param_dist = {
    "n_estimators": [200, 300, 500, 800],
    "max_depth": [4, 6, 8, 10, None],
    "min_samples_leaf": [1, 2, 4, 8],
    "max_features": ["sqrt", "log2", None],
}
rf_search = RandomizedSearchCV(
    RandomForestClassifier(random_state=RANDOM_STATE, class_weight="balanced"),
    rf_param_dist, n_iter=25, scoring="f1_macro", cv=cv,
    random_state=RANDOM_STATE, n_jobs=-1,
)
rf_search.fit(X_train, y_train)
print(f"\nRandom Forest best params: {rf_search.best_params_}")
print(f"Random Forest best CV macro F1: {rf_search.best_score_:.4f}")

xgb_param_dist = {
    "n_estimators": [200, 300, 500],
    "max_depth": [3, 4, 5, 6],
    "learning_rate": [0.01, 0.03, 0.05, 0.1],
    "subsample": [0.7, 0.85, 1.0],
    "colsample_bytree": [0.7, 0.85, 1.0],
}
xgb_search = RandomizedSearchCV(
    XGBClassifier(random_state=RANDOM_STATE, eval_metric="mlogloss"),
    xgb_param_dist, n_iter=25, scoring="f1_macro", cv=cv,
    random_state=RANDOM_STATE, n_jobs=-1,
)
xgb_search.fit(X_train, y_train)
print(f"\nXGBoost best params: {xgb_search.best_params_}")
print(f"XGBoost best CV macro F1: {xgb_search.best_score_:.4f}")

best_rf = rf_search.best_estimator_
best_xgb = xgb_search.best_estimator_

# Compare tuned vs default (default RF from script 02 config) on held-out test set
default_rf = RandomForestClassifier(n_estimators=300, max_depth=8, random_state=RANDOM_STATE,
                                     class_weight="balanced").fit(X_train, y_train)
tuned_test_f1 = f1_score(y_test, best_rf.predict(X_test), average="macro")
default_test_f1 = f1_score(y_test, default_rf.predict(X_test), average="macro")
print(f"\nTest macro F1 - default RF: {default_test_f1:.4f} | tuned RF: {tuned_test_f1:.4f}")

pd.DataFrame([
    {"Model": "Random Forest (default)", "Test Macro F1": default_test_f1},
    {"Model": "Random Forest (tuned)", "Test Macro F1": tuned_test_f1},
]).to_csv("hyperparameter_tuning_comparison.csv", index=False)

# Use the tuned Random Forest as the model of record for all diagnostics below
best_model = best_rf
print("\nUsing tuned Random Forest as the model of record for remaining diagnostics.")


1. HYPERPARAMETER TUNING (RandomizedSearchCV, 5-fold stratified CV)

Random Forest best params: {'n_estimators': 300, 'min_samples_leaf': 4, 'max_features': None, 'max_depth': None}
Random Forest best CV macro F1: 0.4716

XGBoost best params: {'subsample': 0.85, 'n_estimators': 500, 'max_depth': 6, 'learning_rate': 0.1, 'colsample_bytree': 0.85}
XGBoost best CV macro F1: 0.4572

Test macro F1 - default RF: 0.4863 | tuned RF: 0.4935

Using tuned Random Forest as the model of record for remaining diagnostics.


In [6]:
# =======================================================================
# SECTION 2: ROBUSTNESS ANALYSIS (repeated stratified CV)
# =======================================================================
print("\n" + "=" * 70)
print("2. ROBUSTNESS ANALYSIS (Repeated Stratified 5-fold CV, 10 repeats)")
print("=" * 70)

rcv = RepeatedStratifiedKFold(n_splits=5, n_repeats=10, random_state=RANDOM_STATE)
robustness_scores = cross_val_score(best_model, X, y, cv=rcv, scoring="f1_macro", n_jobs=-1)

mean_f1 = robustness_scores.mean()
std_f1 = robustness_scores.std()
ci_low, ci_high = np.percentile(robustness_scores, [2.5, 97.5])
print(f"Macro F1 across 50 folds: mean={mean_f1:.4f}, std={std_f1:.4f}")
print(f"95% percentile interval: [{ci_low:.4f}, {ci_high:.4f}]")

fig, ax = plt.subplots(figsize=(7, 5))
ax.boxplot(robustness_scores, vert=True, patch_artist=True,
           boxprops=dict(facecolor="#2a9d8f", alpha=0.6))
ax.scatter(np.ones(len(robustness_scores)) + np.random.normal(0, 0.02, len(robustness_scores)),
           robustness_scores, alpha=0.4, color="#264653", s=15)
ax.set_ylabel("Macro F1")
ax.set_xticks([1])
ax.set_xticklabels(["Random Forest (tuned)"])
ax.set_title(f"Macro F1 Distribution Across 50 CV Folds\nMean={mean_f1:.3f}, 95% CI=[{ci_low:.3f}, {ci_high:.3f}]")
plt.tight_layout()
plt.savefig("robustness_f1_distribution.png", dpi=150, bbox_inches="tight")
print("Saved: robustness_f1_distribution.png")
plt.close()

pd.DataFrame({"fold_macro_f1": robustness_scores}).to_csv("robustness_cv_scores.csv", index=False)



2. ROBUSTNESS ANALYSIS (Repeated Stratified 5-fold CV, 10 repeats)
Macro F1 across 50 folds: mean=0.4804, std=0.0229
95% percentile interval: [0.4467, 0.5211]
Saved: robustness_f1_distribution.png


In [7]:
# =======================================================================
# SECTION 3: SHAP DEPENDENCE PLOTS
# =======================================================================
print("\n" + "=" * 70)
print("3. SHAP DEPENDENCE PLOTS (top features, Advanced-tier class)")
print("=" * 70)

explainer = shap.TreeExplainer(best_model)
shap_values_raw = explainer.shap_values(X_test)

ADVANCED_CLASS_IDX = 3  # Tier 3 = Advanced; the most policy-relevant class

if isinstance(shap_values_raw, list):
    shap_class = shap_values_raw[ADVANCED_CLASS_IDX]
    mean_abs_shap_all = np.mean([np.abs(sv) for sv in shap_values_raw], axis=0)
elif isinstance(shap_values_raw, np.ndarray) and shap_values_raw.ndim == 3:
    shap_class = shap_values_raw[:, :, ADVANCED_CLASS_IDX]
    mean_abs_shap_all = np.abs(shap_values_raw).mean(axis=2)
else:
    shap_class = shap_values_raw
    mean_abs_shap_all = np.abs(shap_values_raw)

top_features = pd.Series(mean_abs_shap_all.mean(axis=0), index=X.columns) \
    .sort_values(ascending=False).head(4).index.tolist()
print(f"Top 4 features by mean |SHAP|: {top_features}")

fig, axes = plt.subplots(2, 2, figsize=(12, 9))
for ax, feat in zip(axes.flatten(), top_features):
    shap.dependence_plot(feat, shap_class, X_test, interaction_index="auto",
                          ax=ax, show=False)
    ax.set_title(f"SHAP dependence: {feat}\n(Advanced-tier class)")
plt.tight_layout()
plt.savefig("shap_dependence_plots.png", dpi=150, bbox_inches="tight")
print("Saved: shap_dependence_plots.png")
plt.close()



3. SHAP DEPENDENCE PLOTS (top features, Advanced-tier class)
Top 4 features by mean |SHAP|: ['fin17b', 'age', 'in_workforce', 'fin17c']
Saved: shap_dependence_plots.png


In [8]:
# =======================================================================
# SECTION 4: CALIBRATION ANALYSIS
# =======================================================================
print("\n" + "=" * 70)
print("4. CALIBRATION ANALYSIS (per-class reliability diagrams, Brier score)")
print("=" * 70)

y_proba = best_model.predict_proba(X_test)
classes = best_model.classes_

fig, axes = plt.subplots(2, 2, figsize=(11, 9))
brier_scores = {}
for i, (ax, cls) in enumerate(zip(axes.flatten(), classes)):
    y_true_binary = (y_test == cls).astype(int)
    prob_pos = y_proba[:, i]
    brier = brier_score_loss(y_true_binary, prob_pos)
    brier_scores[TIER_NAMES[cls]] = brier

    frac_pos, mean_pred = calibration_curve(y_true_binary, prob_pos, n_bins=5, strategy="quantile")
    ax.plot(mean_pred, frac_pos, marker="o", color="#2a9d8f", label="Model")
    ax.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Perfect calibration")
    ax.set_title(f"{TIER_NAMES[cls]} (Brier={brier:.3f})")
    ax.set_xlabel("Mean predicted probability")
    ax.set_ylabel("Observed frequency")
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig("calibration_reliability_diagrams.png", dpi=150, bbox_inches="tight")
print("Saved: calibration_reliability_diagrams.png")
plt.close()

print("\nBrier scores by tier (lower = better calibrated):")
for tier, score in brier_scores.items():
    print(f"  {tier}: {score:.4f}")
pd.DataFrame([brier_scores]).to_csv("calibration_brier_scores.csv", index=False)


4. CALIBRATION ANALYSIS (per-class reliability diagrams, Brier score)
Saved: calibration_reliability_diagrams.png

Brier scores by tier (lower = better calibrated):
  Excluded: 0.1684
  Basic: 0.0332
  Digitally Engaged: 0.1027
  Advanced: 0.0420


In [9]:
# =======================================================================
# SECTION 5: FEATURE IMPORTANCE CONSISTENCY
# =======================================================================
print("\n" + "=" * 70)
print("5. FEATURE IMPORTANCE CONSISTENCY (Gini vs Permutation vs SHAP)")
print("=" * 70)

gini_importance = pd.Series(best_model.feature_importances_, index=X.columns)

perm_result = permutation_importance(best_model, X_test, y_test, n_repeats=20,
                                      random_state=RANDOM_STATE, scoring="f1_macro", n_jobs=-1)
perm_importance = pd.Series(perm_result.importances_mean, index=X.columns)

shap_importance = pd.Series(mean_abs_shap_all.mean(axis=0), index=X.columns)

importance_df = pd.DataFrame({
    "Gini": gini_importance,
    "Permutation": perm_importance,
    "SHAP": shap_importance,
})
importance_df["Gini_rank"] = importance_df["Gini"].rank(ascending=False)
importance_df["Permutation_rank"] = importance_df["Permutation"].rank(ascending=False)
importance_df["SHAP_rank"] = importance_df["SHAP"].rank(ascending=False)
importance_df = importance_df.sort_values("SHAP", ascending=False)
print(importance_df.round(4).to_string())
importance_df.to_csv("feature_importance_consistency.csv")

rho_gini_shap, p_gini_shap = spearmanr(importance_df["Gini_rank"], importance_df["SHAP_rank"])
rho_perm_shap, p_perm_shap = spearmanr(importance_df["Permutation_rank"], importance_df["SHAP_rank"])
rho_gini_perm, p_gini_perm = spearmanr(importance_df["Gini_rank"], importance_df["Permutation_rank"])
print(f"\nSpearman rank correlation:")
print(f"  Gini vs SHAP: rho={rho_gini_shap:.3f}, p={p_gini_shap:.4f}")
print(f"  Permutation vs SHAP: rho={rho_perm_shap:.3f}, p={p_perm_shap:.4f}")
print(f"  Gini vs Permutation: rho={rho_gini_perm:.3f}, p={p_gini_perm:.4f}")

fig, ax = plt.subplots(figsize=(9, 6))
importance_df[["Gini", "Permutation", "SHAP"]].apply(
    lambda col: col / col.max()).plot(kind="barh", ax=ax, width=0.75)
ax.set_xlabel("Normalized importance (0-1 within method)")
ax.set_title("Feature Importance Consistency Across Three Methods")
ax.invert_yaxis()
plt.tight_layout()
plt.savefig("feature_importance_consistency.png", dpi=150, bbox_inches="tight")
print("Saved: feature_importance_consistency.png")
plt.close()


5. FEATURE IMPORTANCE CONSISTENCY (Gini vs Permutation vs SHAP)
                   Gini  Permutation    SHAP  Gini_rank  Permutation_rank  SHAP_rank
fin17b           0.2099       0.1087  0.0915        2.0               1.0        1.0
age              0.2307       0.0059  0.0571        1.0               7.0        2.0
in_workforce     0.0786       0.0221  0.0551        5.0               3.0        3.0
fin17c           0.0689       0.0459  0.0364        6.0               2.0        4.0
inc_q            0.0905       0.0067  0.0319        3.0               6.0        5.0
con1             0.0437      -0.0108  0.0242        8.0              14.0        6.0
fin24b           0.0808       0.0105  0.0211        4.0               5.0        7.0
female           0.0306      -0.0169  0.0191       11.0              15.0        8.0
internet_use     0.0349      -0.0077  0.0187        9.0              12.0        9.0
fin24a           0.0486      -0.0031  0.0180        7.0              10.0       10.0


In [10]:
# =======================================================================
# SECTION 6: ERROR ANALYSIS
# =======================================================================
print("\n" + "=" * 70)
print("6. ERROR ANALYSIS (confusion pairs, subgroup error rates)")
print("=" * 70)

y_pred = best_model.predict(X_test)
cm = confusion_matrix(y_test, y_pred)
print("Confusion matrix (rows=true, cols=predicted):")
print(pd.DataFrame(cm, index=TIER_NAMES, columns=TIER_NAMES).to_string())

# Largest off-diagonal confusion pair
cm_no_diag = cm.copy().astype(float)
np.fill_diagonal(cm_no_diag, 0)
worst_true, worst_pred = np.unravel_index(cm_no_diag.argmax(), cm_no_diag.shape)
print(f"\nLargest confusion pair: true={TIER_NAMES[worst_true]}, "
      f"predicted={TIER_NAMES[worst_pred]}, count={int(cm_no_diag[worst_true, worst_pred])}")

# Subgroup error rates (also feeds the Ethics/fairness discussion)
results_test = X_test.copy()
results_test["y_true"] = y_test.values
results_test["y_pred"] = y_pred
results_test["correct"] = (results_test["y_true"] == results_test["y_pred"])

print("\nAccuracy by gender:")
acc_by_gender = results_test.groupby("female")["correct"].mean()
acc_by_gender.index = ["Male", "Female"]
print(acc_by_gender.round(4).to_string())

print("\nAccuracy by urban/rural:")
acc_by_rural = results_test.groupby("is_rural")["correct"].mean()
acc_by_rural.index = ["Urban", "Rural"]
print(acc_by_rural.round(4).to_string())

# Disparate impact style ratio: min/max of subgroup accuracy (closer to 1.0 = more equitable)
gender_di = acc_by_gender.min() / acc_by_gender.max()
rural_di = acc_by_rural.min() / acc_by_rural.max()
print(f"\nSubgroup accuracy ratio (min/max, closer to 1.0 = more equitable):")
print(f"  Gender: {gender_di:.3f}")
print(f"  Urban/Rural: {rural_di:.3f}")

pd.DataFrame({
    "subgroup_type": ["gender", "gender", "rural", "rural"],
    "subgroup": ["Male", "Female", "Urban", "Rural"],
    "accuracy": [acc_by_gender["Male"], acc_by_gender["Female"],
                 acc_by_rural["Urban"], acc_by_rural["Rural"]],
}).to_csv("error_analysis_subgroup_accuracy.csv", index=False)

# Characterize the worst confusion pair: mean feature values, misclassified vs correct
mask_pair = (results_test["y_true"] == worst_true)
pair_subset = results_test[mask_pair].copy()
pair_subset["classified_correctly"] = pair_subset["y_true"] == pair_subset["y_pred"]
compare_cols = [c for c in X.columns]
pair_profile = pair_subset.groupby("classified_correctly")[compare_cols].mean().round(3)
print(f"\nFeature means for true={TIER_NAMES[worst_true]}, correct vs misclassified:")
print(pair_profile.to_string())
pair_profile.to_csv("error_analysis_worst_pair_profile.csv")


6. ERROR ANALYSIS (confusion pairs, subgroup error rates)
Confusion matrix (rows=true, cols=predicted):
                   Excluded  Basic  Digitally Engaged  Advanced
Excluded                140      3                 21         7
Basic                     3      0                  2         0
Digitally Engaged        16      0                  8         1
Advanced                  4      0                  2        43

Largest confusion pair: true=Excluded, predicted=Digitally Engaged, count=21

Accuracy by gender:
Male      0.6889
Female    0.8522

Accuracy by urban/rural:
Urban    0.7518
Rural    0.7798

Subgroup accuracy ratio (min/max, closer to 1.0 = more equitable):
  Gender: 0.808
  Urban/Rural: 0.964

Feature means for true=Excluded, correct vs misclassified:
                      female     age   educ  inc_q  in_workforce  is_rural   con1  internet_use  merchantpay_dig  fin17b  fin17c  borrowed  fin22a_1  fin24a  fin24b
classified_correctly                                  

In [11]:
# =======================================================================
# SECTION 7: ABLATION STUDY
# =======================================================================
print("\n" + "=" * 70)
print("7. ABLATION STUDY (feature-group leave-one-out and single-group-only)")
print("=" * 70)

def eval_feature_subset(cols, label):
    Xs_train, Xs_test = X_train[cols], X_test[cols]
    model = RandomForestClassifier(**{k: v for k, v in rf_search.best_params_.items()},
                                    random_state=RANDOM_STATE, class_weight="balanced")
    model.fit(Xs_train, y_train)
    preds = model.predict(Xs_test)
    f1 = f1_score(y_test, preds, average="macro")
    acc = accuracy_score(y_test, preds)
    print(f"  {label:35s} n_features={len(cols):2d}  macro F1={f1:.4f}  accuracy={acc:.4f}")
    return {"configuration": label, "n_features": len(cols), "macro_f1": f1, "accuracy": acc}

ablation_results = []
all_features = list(X.columns)
ablation_results.append(eval_feature_subset(all_features, "Full feature set"))

print("\nLeave-one-group-out:")
for group_name, group_cols in feature_groups.items():
    remaining = [c for c in all_features if c not in group_cols]
    ablation_results.append(eval_feature_subset(remaining, f"All except {group_name}"))

print("\nSingle-group-only:")
for group_name, group_cols in feature_groups.items():
    ablation_results.append(eval_feature_subset(group_cols, f"{group_name} only"))

ablation_df = pd.DataFrame(ablation_results)
ablation_df.to_csv("ablation_study_results.csv", index=False)

fig, ax = plt.subplots(figsize=(10, 6))
colors = ["#264653"] + ["#e76f51"] * len(feature_groups) + ["#2a9d8f"] * len(feature_groups)
ax.barh(ablation_df["configuration"], ablation_df["macro_f1"], color=colors)
ax.set_xlabel("Macro F1")
ax.set_title("Ablation Study: Feature-Group Contribution to Macro F1")
ax.invert_yaxis()
plt.tight_layout()
plt.savefig("ablation_study.png", dpi=150, bbox_inches="tight")
print("\nSaved: ablation_study.png")
plt.close()

print("\n" + "=" * 70)
print("Model diagnostics complete. Files saved:")
print("  - hyperparameter_tuning_comparison.csv")
print("  - robustness_f1_distribution.png / robustness_cv_scores.csv")
print("  - shap_dependence_plots.png")
print("  - calibration_reliability_diagrams.png / calibration_brier_scores.csv")
print("  - feature_importance_consistency.png / .csv")
print("  - error_analysis_subgroup_accuracy.csv / _worst_pair_profile.csv")
print("  - ablation_study.png / ablation_study_results.csv")
print("=" * 70)


7. ABLATION STUDY (feature-group leave-one-out and single-group-only)
  Full feature set                    n_features=15  macro F1=0.4935  accuracy=0.7640

Leave-one-group-out:
  All except demographics             n_features= 9  macro F1=0.4798  accuracy=0.6360
  All except access                   n_features=13  macro F1=0.5060  accuracy=0.7600
  All except behaviour                n_features=10  macro F1=0.4148  accuracy=0.6800
  All except resilience               n_features=13  macro F1=0.4894  accuracy=0.7280

Single-group-only:
  demographics only                   n_features= 6  macro F1=0.3509  accuracy=0.6040
  access only                         n_features= 2  macro F1=0.1860  accuracy=0.2080
  behaviour only                      n_features= 5  macro F1=0.3888  accuracy=0.4840
  resilience only                     n_features= 2  macro F1=0.3053  accuracy=0.4560

Saved: ablation_study.png

Model diagnostics complete. Files saved:
  - hyperparameter_tuning_comparison.csv
  -